<a href="https://colab.research.google.com/github/Ans365332/6may-file-example/blob/main/RAG_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# RAG System — ChromaDB + Gemini API
# This notebook builds a minimal Retrieval-Augmented Generation (RAG) pipeline:

# Install dependencies
# Set up Gemini API key
# Load & chunk documents
# Create embeddings and store them in ChromaDB
# Retrieve relevant chunks for a user query
# Generate the final answer using Gemini
# Runs end-to-end on Google Colab (free tier, CPU is enough).

#Step 1: Install Dependencies

In [2]:
# Install required libraries
# chromadb -> vector database to store & search embeddings
# google-generativeai -> official Gemini API SDK
!pip install -q chromadb google-generativeai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 760.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

#Step 2: Set Up Gemini API Key
Get a free API key from https://aistudio.google.com/app/apikey

We use Colab's userdata (Secrets) so the key is not hardcoded in the notebook. Click the 🔑 icon in the left sidebar of Colab → add a secret named GEMINI_API_KEY.

If you are not using Colab Secrets, you can uncomment the fallback line and paste your key directly (not recommended for sharing).





In [9]:
import google.generativeai as genai
from google.colab import userdata

import google.generativeai as genai
from google.colab import userdata

# Try to read the key from Colab Secrets first
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = None

# Fallback: uncomment and paste your key here if not using Colab Secrets
# GEMINI_API_KEY = "YOUR_API_KEY_HERE"

if not GEMINI_API_KEY:
    raise ValueError("Please set GEMINI_API_KEY in Colab Secrets or paste it directly above.")

# Configure the Gemini client with our API key
genai.configure(api_key=GEMINI_API_KEY)

print("Gemini API configured successfully")


Gemini API configured successfully


#Step 3: Load Documents
For demo purposes we use a small list of text documents. Replace this with your own data (PDF text, articles, notes, etc.).

In [10]:
# Sample knowledge base (replace this with your own documents)
documents = [
    "ASCA Trainer is an AI/ML and GenAI training academy that builds curriculum for interns and IT students.",
    "Retrieval-Augmented Generation (RAG) combines a retriever (vector search) with a generator (LLM) to answer questions using external knowledge.",
    "ChromaDB is an open-source vector database used to store embeddings and perform similarity search efficiently.",
    "Gemini is Google's family of large language models, accessible via the google-generativeai Python SDK.",
    "FastAPI is a modern, high-performance Python web framework commonly used to build backend APIs for AI applications.",
    "Docker allows packaging an application with all its dependencies into a container that can run consistently on any machine.",
    "AWS EC2 provides scalable virtual servers in the cloud, commonly used to deploy backend services and AI applications.",
]

print(f"Loaded {len(documents)} documents")


Loaded 7 documents


#Step 4: Chunk the Text
Long documents should be split into smaller chunks before embedding, so retrieval is more precise. For this small demo each document is already short, but the function below works for longer text too.

In [11]:
def chunk_text(text, chunk_size=300, overlap=50):
  """
  Split a long text into overlapping chunks.
  chunk_size -> max characters per chunk
  overlap  -> characters repeated between consecutive chunks (keeps context continuity )
  """
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start += chunk_size - overlap

  return chunks

# Apply chunking to every documents and keep track of source document id
all_chunks = []
metadata = []

for doc_id, doc in enumerate(documents):
  doc_chunks = chunk_text(doc)
  for chunk in doc_chunks:
    all_chunks.append(chunk)
    metadata.append({"source_doc_id": doc_id})

print(f"Total chunks created: {len(all_chunks)}")




Total chunks created: 7


In [12]:
!pip install sentence-transformers

#Step 5: Create Embeddings & Store in ChromaDB
We use Gemini's embedding model (text-embedding-004) to convert each chunk into a vector, then store the vectors in a local ChromaDB collection.




In [13]:
import chromadb
from sentence_transformers import SentenceTransformer

# --------------------------------------------------
# 1. Load Hugging Face embedding model
# --------------------------------------------------
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


# --------------------------------------------------
# 2. Create an in-memory ChromaDB client
# --------------------------------------------------
chroma_client = chromadb.Client()

collection_name = "rag_demo_collection"

# Delete existing collection if it exists
try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass

# Create collection
collection = chroma_client.create_collection(
    name=collection_name
)


# --------------------------------------------------
# 3. Embedding function
# --------------------------------------------------
def embed_text(text):
    """
    Generate an embedding using Hugging Face
    all-MiniLM-L6-v2
    """
    embedding = embedding_model.encode(
        text,
        convert_to_numpy=True
    )

    return embedding.tolist()


# --------------------------------------------------
# 4. Generate embeddings for all document chunks
# --------------------------------------------------
ids = [str(i) for i in range(len(all_chunks))]

embeddings = [
    embed_text(chunk)
    for chunk in all_chunks
]


# --------------------------------------------------
# 5. Store chunks + embeddings in ChromaDB
# --------------------------------------------------
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=all_chunks,
    metadatas=metadata
)


# --------------------------------------------------
# 6. Check how many chunks were stored
# --------------------------------------------------
print(f"Stored {collection.count()} chunks in ChromaDB")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stored 7 chunks in ChromaDB


#Step 6: Retrieve Relevant Chunks for a Query

In [14]:
def retrieve_context(query, top_k=3):
    """
    Embed the user query using all-MiniLM-L6-v2
    and fetch the top_k most similar chunks from ChromaDB.
    """

    # Generate query embedding
    query_embedding = embed_text(query)

    # Search ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    # Get retrieved documents
    retrieved_chunks = results["documents"][0]

    return retrieved_chunks


# --------------------------------------------------
# Quick test of retrieval only
# --------------------------------------------------

test_query = "What is ChromaDB used for?"

retrieved = retrieve_context(test_query)

for i, chunk in enumerate(retrieved, 1):
    print(f"[{i}] {chunk}\n")

[1] ChromaDB is an open-source vector database used to store embeddings and perform similarity search efficiently.

[2] FastAPI is a modern, high-performance Python web framework commonly used to build backend APIs for AI applications.

[3] Gemini is Google's family of large language models, accessible via the google-generativeai Python SDK.



#Step 7: Generate the Final Answer with Gemini
We pass the retrieved chunks as context along with the user's question to Gemini, and ask it to answer strictly based on that context.

In [18]:
# Load the Gemini generation model
generation_model = genai.GenerativeModel("gemini-3.6-flash")


def generate_answer(query, top_k=3):
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks from ChromaDB
    2. Build a prompt with context + question
    3. Ask Gemini to answer using only that context
    """
    context_chunks = retrieve_context(query, top_k=top_k)
    context_text = "\n\n".join(context_chunks)

    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not present in the context, say "I don't have enough information to answer that."

Context:
{context_text}

Question: {query}

Answer:"""

    response = generation_model.generate_content(prompt)
    return response.text, context_chunks


# Run the full pipeline on a sample question
answer, used_context = generate_answer("What is RAG and how does ChromaDB help?")

print("ANSWER:\n", answer)
print("\n--- Context used ---")
for c in used_context:
    print("-", c)


ANSWER:
 Retrieval-Augmented Generation (RAG) combines a retriever (vector search) with a generator (LLM) to answer questions using external knowledge. 

ChromaDB helps by acting as an open-source vector database that is used to store embeddings and perform similarity searches efficiently.

--- Context used ---
- ChromaDB is an open-source vector database used to store embeddings and perform similarity search efficiently.
- Retrieval-Augmented Generation (RAG) combines a retriever (vector search) with a generator (LLM) to answer questions using external knowledge.
- FastAPI is a modern, high-performance Python web framework commonly used to build backend APIs for AI applications.


#Step 8: Ask Your Own Questions
Run this cell as many times as you like with different questions.

In [19]:
user_question = "How does Docker help in deploying AI applications?"  # change this to any question

answer, used_context = generate_answer(user_question)
print("Q:", user_question)
print("\nA:", answer)

Q: How does Docker help in deploying AI applications?

A: Based on the context provided, Docker helps by allowing an application to be packaged with all its dependencies into a container that can run consistently on any machine.


In [20]:
user_question = "How Blinkit or Zomato works?"  # change this to any question

answer, used_context = generate_answer(user_question)
print("Q:", user_question)
print("\nA:", answer)

Q: How Blinkit or Zomato works?

A: I don't have enough information to answer that.


In [21]:
# Documents ---> Chunking ---> Embedding(Huggingface) ----> VD ChromaDB ---> Collection create--->
# All embeddings store --->
# User Query ---> VD Send ---> Relevent chunks documents return

# LLM Setup ---> gemini api key , model
# Prompt
# User question ---> Embedding ---> vd ---> related chunks ---> documents --> LLM Return
# IF related chunks not found ----> LLM Return 'Don't Know'